# PyroSync AOI Insights through the API (example)

This notebook reads the **AOI Insights** endpoints of a running PyroSync (version 1.3) and reproduces the main numbers of the tab.
Start the app first (`start` script or `python -m pyrosync.cli serve`), import your FIRMS archive, then run the cells.

Everything here describes **observed satellite detections** for one area and one sensor series. It is not a forecast, not a fire-danger rating and not an official warning.
Needs only `requests` (and optionally `pandas` and `matplotlib` for the table and chart).

In [ ]:
import requests

BASE = "http://127.0.0.1:8000"      # change if the app runs elsewhere (the online demo is read-only but works too)
AOI = "region:kalimantan"
SERIES = "MODIS"                     # MODIS, VIIRS_SNPP, VIIRS_N20 or VIIRS_N21 (never added together)
PLATFORMS = {"MODIS": "Terra,Aqua", "VIIRS_SNPP": "SNPP", "VIIRS_N20": "N20", "VIIRS_N21": "N21"}[SERIES]

inv = requests.get(f"{BASE}/api/inventory", timeout=60).json()
plats = PLATFORMS.split(",")
AS_OF = max(p["last_date"] for p in inv["products"] if p["platform"] in plats)
print("as of", AS_OF, "| history starts", inv["date_range"][0])

def q(start=None, end=AS_OF, **extra):
    p = {"start": start or f"{end[:4]}-01-01", "end": end, "aoi": AOI, "platforms": PLATFORMS, "series": SERIES}
    p.update(extra)
    return p

def get(path, **params):
    r = requests.get(f"{BASE}{path}", params=params, timeout=300)
    r.raise_for_status()
    return r.json()

## Season tracker and fire regime

In [ ]:
reg = get("/api/insights/regime", **q(metric="detections"))
if not reg["available"]:
    print("Not enough history:", reg["reason"])
else:
    print(reg["tracker"]["status"]["sentence"])
    s = reg["season"]
    print("Season start / peak / end (median):", s["onset"]["median"], "/", s["peak"]["median"], "/", s["end"]["median"])
    print("Busiest week:", reg["peak_week"]["label"], "| CV of annual totals:", round(reg["variability"]["cv"], 2))
    print(reg["disclaimer"])

In [ ]:
# optional chart: cumulative detections of the focus year vs the reference envelope
try:
    import matplotlib.pyplot as plt
    if reg["available"]:
        e, x = reg["tracker"]["envelope"], range(1, 366)
        plt.fill_between(x, e["p10"], e["p90"], alpha=.25, label="10th-90th percentile of reference years")
        plt.plot(x, e["median"], "--", label="median")
        plt.plot(range(1, len(reg["tracker"]["focus"]) + 1), reg["tracker"]["focus"], label=str(reg["focus_year"]))
        plt.xlabel("day of year"); plt.ylabel("cumulative detections"); plt.legend(); plt.show()
except ImportError:
    print("matplotlib not installed: skipping the chart")

## Unusual conditions (robust score, last 180 days)

In [ ]:
from datetime import date, timedelta
start = max(inv["date_range"][0], (date.fromisoformat(AS_OF) - timedelta(days=179)).isoformat())
an = get("/api/insights/anomaly", **q(start=start, metric="detections"))
print(an["summary"]["by_band"])
m = an["summary"]["max"]
if m:
    print("most unusual day:", m["date"], m["value"], "detections, usual median", m["reference_median"], "->", m["band"])
print(an["method"]["score"])

## Critical periods and readiness calendar

In [ ]:
cr = get("/api/insights/critical", **q(metric="detections", percentile=90))
if cr["available"]:
    for p in cr["periods"][:5]:
        print(p["rank"], p["start"], "to", p["end"], p["duration_days"], "days, peak", p["peak"])
    top = sorted(cr["readiness"]["weeks"], key=lambda w: -(w["share"] or 0))[:4]
    print("most often critical weeks:", [(w["label"], f"{w['years_critical']}/{w['years_known']}") for w in top])
else:
    print(cr["reason"])

## Years, ENSO/IOD phases (analog years) and recurrence

In [ ]:
yr = get("/api/insights/years", **q(metric="detections"))
for g in yr["groups"]["enso"]:
    print(f"{g['phase']:>8}: {g['n_years']} years, median detections {g['median_total']}")
rec = get("/api/insights/recurrence", **q(chronic_share=0.4))
if rec["available"]:
    print(rec["chronic_cells"], "chronic cells of", rec["cells_total"], "cells with fire in", rec["n_eligible_years"], "complete years")
try:
    import pandas as pd
    display(pd.DataFrame(yr["years"])[["year", "eligible", "total", "critical_days", "enso", "oni", "iod", "dmi"]])
except ImportError:
    pass

## Download the reproducible package

The same files as **Method & data → Download the insights package** and `python -m pyrosync.cli insights`.

In [ ]:
r = requests.get(f"{BASE}/api/export/insights.zip", params=q(metric="detections"), timeout=600)
r.raise_for_status()
fn = f"pyrosync_insights_{SERIES}_{AS_OF}.zip"
open(fn, "wb").write(r.content)
print("wrote", fn, len(r.content), "bytes")